# HOUSEKEEPING

In [1]:
import sys
import os
sys.path.append('/scratch_net/ken/radjoe/Projects/Experiments/SAMEXP/')

In [2]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats 
from scipy.stats import shapiro , kstest, mannwhitneyu, ttest_rel
import torch
from torch import nn
from datasets.datasets import MRIDataset, STAREDataset

from helper_func.analyis_helper import clean_df, plot_boxplots, visualize, generate_batch_views, visualize_views

# MAIN

## Loads

In [42]:
TTA_NA_path = "/scratch_net/ken/radjoe/Projects/Experiments/SAMEXP/results/TTA_stuff/SSA_NA_ent.csv"
TTA_A_path = "/scratch_net/ken/radjoe/Projects/Experiments/SAMEXP/results/TTA_stuff/SSA_A_ent.csv"
SSA_NA_path = "/scratch_net/ken/radjoe/Projects/Experiments/SAMEXP/results/TT_NA_SSA/boxes/SSA_boxes.csv"
SSA_A_path = "/scratch_net/ken/radjoe/Projects/Experiments/SAMEXP/results/TT_A_SSA/boxes/SSA_boxes.csv"

In [60]:
SSA_NA_df = pd.read_csv(SSA_NA_path)
SSA_NA_df = clean_df(SSA_NA_df, True, 'brats')
SSA_NA_df

,name,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
0,BraTS-SSA-00002-000.npy,0.405957,0.881102,0.728119,37.357056,12.884099,72.365730,0.899894,0.893997,0.639472,0.262097,0.868574,0.845299
1,BraTS-SSA-00012-000.npy,0.860987,0.873251,0.703306,3.605551,48.228622,62.091831,0.994836,0.975560,0.753016,0.758884,0.790365,0.659753
2,BraTS-SSA-00049-000.npy,0.872074,0.769765,0.850662,2.236068,42.673176,58.083569,0.860781,0.952165,0.898448,0.883666,0.646013,0.807703
3,BraTS-SSA-00096-000.npy,0.812003,0.578594,0.654730,10.246951,15.308474,13.003844,0.772676,0.465267,0.541164,0.855547,0.764903,0.828622
4,BraTS-SSA-00113-000.npy,0.883940,0.792710,0.644667,4.577075,22.293497,16.278820,0.827667,0.781040,0.803804,0.948424,0.804735,0.538129
5,BraTS-SSA-00115-000.npy,0.628451,0.277822,0.466160,6.082763,46.043457,41.484936,0.603681,0.698824,0.358492,0.655341,0.173374,0.666261
6,BraTS-SSA-00119-000.npy,0.332665,0.636030,0.675059,45.822964,64.292679,48.533493,0.384259,0.599577,0.663842,0.293286,0.677202,0.686662
7,BraTS-SSA-00152-000.npy,0.769118,0.725482,0.885938,5.385165,43.692104,8.602325,0.698904,0.821422,0.880933,0.855014,0.649610,0.891000
8,BraTS-SSA-00223-000.npy,0.187597,0.884516,0.717911,8.544003,12.727922,5.830952,0.108287,0.967102,0.584101,0.701043,0.814925,0.931247


In [44]:
SSA_A_df = pd.read_csv(SSA_A_path)
SSA_A_df = clean_df(SSA_A_df, True, 'brats')
SSA_A_df

,name,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
0,BraTS-SSA-00002-000.npy,0.559861,0.901478,0.656419,8.544003,21.954498,5.744563,0.978040,0.870486,0.515843,0.392178,0.934758,0.902316
1,BraTS-SSA-00012-000.npy,0.836975,0.856495,0.718330,5.196152,43.665207,39.971199,0.991178,0.857724,0.581433,0.724292,0.855270,0.939541
2,BraTS-SSA-00049-000.npy,0.850239,0.802588,0.846821,2.828427,48.769348,58.403767,0.787952,0.842691,0.827064,0.923219,0.766128,0.867544
3,BraTS-SSA-00096-000.npy,0.733146,0.345016,0.742987,20.571823,41.243183,10.630146,0.730546,0.243778,0.726868,0.735765,0.590061,0.759838
4,BraTS-SSA-00113-000.npy,0.754802,0.776796,0.611024,47.520000,50.575169,16.982157,0.767884,0.783401,0.491701,0.742158,0.770302,0.806818
5,BraTS-SSA-00115-000.npy,0.616795,0.293922,0.465482,8.246211,44.944408,17.501390,0.739748,0.477532,0.335355,0.528889,0.212295,0.760629
6,BraTS-SSA-00119-000.npy,0.320641,0.566169,0.543444,8.306623,39.458839,44.269630,0.370370,0.506927,0.432868,0.282686,0.641089,0.729896
7,BraTS-SSA-00152-000.npy,0.814407,0.734292,0.859619,3.162278,29.427877,5.477226,0.879401,0.741690,0.792703,0.758359,0.727041,0.938874
8,BraTS-SSA-00223-000.npy,0.206508,0.877770,0.668171,34.856842,31.968735,8.062258,0.127797,0.906144,0.526743,0.537651,0.851120,0.913422


In [47]:
TTA_A_df = pd.read_csv(TTA_A_path)
TTA_A_df = clean_df(TTA_A_df, True, 'brats')
TTA_A_df

,name,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
0,BraTS-SSA-00012-000.npy,0.860799,0.782983,0.601267,2.449490,46.195236,7.000000,0.922754,0.855983,0.458874,0.806640,0.721455,0.871791
1,BraTS-SSA-00113-000.npy,0.387550,0.615206,0.035605,18.478365,11.090536,36.400551,0.244769,0.760741,0.018177,0.930111,0.516413,0.863333
2,BraTS-SSA-00152-000.npy,0.070334,0.044183,0.259166,27.037012,41.701321,21.470911,0.036469,0.022630,0.155035,0.985178,0.928258,0.789344
3,BraTS-SSA-00223-000.npy,0.000000,0.861160,0.305115,23.008690,15.937377,12.409674,0.000000,0.899591,0.180605,0.000000,0.825879,0.982351


In [46]:
TTA_NA_df = pd.read_csv(TTA_NA_path)
TTA_NA_df = clean_df(TTA_NA_df, True, 'brats')
TTA_NA_df.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000
mean,0.333316,0.553450,0.329680,33.373044,35.839671,36.973480,0.420687,0.568352,0.248397,0.595213,0.707378,0.683631
std,0.370324,0.354675,0.272095,26.212550,11.500056,41.854204,0.461075,0.357328,0.238094,0.393567,0.295311,0.339421
min,0.000000,0.019878,0.000000,6.403124,23.667454,6.000000,0.000000,0.010039,0.000000,0.000000,0.174497,0.000000
25%,0.029666,0.311977,0.106617,19.482913,27.154945,10.599269,0.015064,0.355515,0.060550,0.361216,0.627350,0.757935
50%,0.231631,0.730337,0.369389,20.926748,32.992256,22.961182,0.334468,0.752758,0.241798,0.670792,0.810561,0.796216
75%,0.585628,0.811357,0.481159,53.647838,45.888228,40.502604,0.819299,0.804807,0.337031,0.909131,0.867737,0.840904
max,0.870661,0.829264,0.701859,68.425140,49.989998,117.090141,0.965680,0.849300,0.637879,0.983092,0.995575,0.908378


### Consistency

In [57]:
TTA_NA_path = "/scratch_net/ken/radjoe/Projects/Experiments/SAMEXP/logs/tta_feature_check/logs/BraTS_GLI/SSA_boxes.csv"
TTA_A_path = "/scratch_net/ken/radjoe/Projects/Experiments/SAMEXP/logs/tta_feature_check_augs/logs/BraTS_GLI/SSA_boxes.csv"

In [58]:
TTA_A_df = pd.read_csv(TTA_A_path)
TTA_A_df = clean_df(TTA_A_df, True, 'brats')
TTA_A_df.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,7.000000,7.000000,7.000000,7.000000,7.000000,7.000000,7.000000,7.000000,7.000000,7.000000,7.000000,7.000000
mean,0.298417,0.492906,0.240113,32.053527,28.054615,25.793385,0.273828,0.651318,0.162463,0.680319,0.645428,0.900419
std,0.371303,0.355010,0.231194,35.228529,14.620011,17.245323,0.382884,0.421179,0.178304,0.375123,0.262841,0.074136
min,0.000000,0.066258,0.007114,2.449490,10.816654,6.633250,0.000000,0.034695,0.003572,0.000000,0.125277,0.801551
25%,0.003170,0.151081,0.050584,13.368159,18.086672,13.911116,0.001592,0.422599,0.026030,0.548225,0.574327,0.852445
50%,0.075431,0.658393,0.184883,19.104973,26.115129,17.117243,0.039340,0.891835,0.102165,0.825114,0.734000,0.867089
75%,0.574933,0.784759,0.379527,35.139213,37.654219,38.093515,0.462956,0.910092,0.241443,0.925827,0.804778,0.972247
max,0.857282,0.854009,0.628571,105.805481,47.968739,52.793938,0.948359,0.967316,0.496555,0.989011,0.900507,0.984914


In [59]:
TTA_NA_df = pd.read_csv(TTA_NA_path)
TTA_NA_df = clean_df(TTA_NA_df, True, 'brats')
TTA_NA_df.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000
mean,0.312754,0.542675,0.349551,36.438896,41.283891,39.457582,0.437238,0.732088,0.290869,0.557585,0.598040,0.620449
std,0.348312,0.317688,0.276314,19.565204,9.664798,40.920521,0.474424,0.360750,0.278945,0.393091,0.287323,0.309744
min,0.000000,0.086137,0.000000,18.411953,26.172504,6.782330,0.000000,0.045082,0.000000,0.000000,0.120094,0.000000
25%,0.022806,0.309856,0.122220,21.265974,36.806349,13.757725,0.011691,0.683232,0.073960,0.305566,0.509783,0.668282
50%,0.230764,0.680142,0.411502,31.291258,41.663507,27.740023,0.365303,0.896933,0.281271,0.592261,0.617159,0.741817
75%,0.516566,0.767196,0.517882,47.214363,48.091345,42.738628,0.858088,0.954216,0.389205,0.874301,0.750939,0.757923
max,0.849675,0.826989,0.695166,67.192627,52.782574,117.647995,0.976654,0.968270,0.752636,0.982675,0.964306,0.831999


In [ ]:
# Problems are the boundaries and also the sam_tokens have a lot of domain shift

In [61]:
TTA_NA_path = "/scratch_net/ken/radjoe/Projects/Experiments/SAMEXP/logs/tta_feature_check/logs/BraTS_GLI/SSA_boxes.csv"
TTA_A_path = "/scratch_net/ken/radjoe/Projects/Experiments/SAMEXP/logs/tta_feature_check_augs/logs/BraTS_GLI/SSA_boxes.csv"

In [62]:
TTA_A_df = pd.read_csv(TTA_A_path)
TTA_A_df = clean_df(TTA_A_df, True, 'brats')
TTA_A_df.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000
mean,0.699499,0.069527,0.082703,12.727922,27.535416,16.881943,0.613231,0.03618,0.043175,0.814012,0.888114,0.978827
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,0.699499,0.069527,0.082703,12.727922,27.535416,16.881943,0.613231,0.03618,0.043175,0.814012,0.888114,0.978827
25%,0.699499,0.069527,0.082703,12.727922,27.535416,16.881943,0.613231,0.03618,0.043175,0.814012,0.888114,0.978827
50%,0.699499,0.069527,0.082703,12.727922,27.535416,16.881943,0.613231,0.03618,0.043175,0.814012,0.888114,0.978827
75%,0.699499,0.069527,0.082703,12.727922,27.535416,16.881943,0.613231,0.03618,0.043175,0.814012,0.888114,0.978827
max,0.699499,0.069527,0.082703,12.727922,27.535416,16.881943,0.613231,0.03618,0.043175,0.814012,0.888114,0.978827


In [63]:
TTA_NA_df = pd.read_csv(TTA_NA_path)
TTA_NA_df = clean_df(TTA_NA_df, True, 'brats')
TTA_NA_df.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000
mean,0.323274,0.557050,0.343095,36.465114,40.421482,39.341071,0.430900,0.678076,0.272454,0.571964,0.647378,0.647802
std,0.361781,0.348117,0.279898,19.128094,9.392224,40.876354,0.470970,0.362220,0.261286,0.387390,0.293992,0.318732
min,0.000000,0.038905,0.000000,18.973665,28.017851,6.000000,0.000000,0.019846,0.000000,0.000000,0.125122,0.000000
25%,0.024075,0.315520,0.110718,20.987010,34.262730,14.213308,0.012215,0.577232,0.065229,0.332870,0.580256,0.737214
50%,0.225974,0.726883,0.395604,31.181722,39.964899,27.811276,0.351486,0.850842,0.270237,0.632306,0.708592,0.778296
75%,0.560885,0.811669,0.503257,49.355385,47.119198,42.405181,0.846328,0.903451,0.365521,0.871408,0.794414,0.787899
max,0.858400,0.831299,0.712465,64.451530,52.715275,117.473404,0.972028,0.939910,0.700072,0.983005,0.981465,0.815601
